In [1]:
## connessione al google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy  as np
import math

#from ape_dataset import *

import os

# cartella output per i csv di noise
OUTDIR = "csv_reco"
os.makedirs(OUTDIR, exist_ok=True)

In [3]:
## DATASET GENERATION ###

##Defines
c_x = 175  #center of sector division (local coords)
c_y = 189

c1 = 680   #line interception with y axis
c2 = 320

grid_size = 8 #grid resolution (preprocessing step)

num_noise_ROOT_files = 100 #number of input files
num_signal_ROOT_files = 100 #number of input files
num_events = 1000 #number of events for each file

num_sectors = 6 #number of sectors of dRICH


In [4]:
binwidth=10.16/10
ten=10.16
nbins=int(ten/binwidth)

In [5]:

def noise_probability_gaussian(freq_kHz=100,time_window_ns=10):
    p = freq_kHz*1000 *time_window_ns/1000000000
    sigma_p = p * 0.01
    single_ph_emission_p = np.random.normal(0, 1) * sigma_p + p
    if single_ph_emission_p < 0:
        single_ph_emission_p = 0

    return single_ph_emission_p


def neq_radius(neq_radius_params,radius):
    neq = np.zeros(radius.size)
    for i,par in enumerate(neq_radius_params):
        neq+=neq_radius_params[i]*pow(radius,i)
    return neq


def noise_probability_radial(luminosity_fbi=100,time_window_ns=10,radius=[]):
    baseline_dcr = 3000*np.ones(radius.size)
    neq_radius_params = [-3.27029e+09, 1.26055e+08, -1.88568e+06, 13929.1, -50.9931, 0.0741068]
    dcr_increase = 300/1000000
    neq = neq_radius(neq_radius_params,radius) * luminosity_fbi;
    dcr = baseline_dcr + dcr_increase * neq
    pro = dcr * time_window_ns/1000000000;

    return pro



def pseudorapidity(px, py, pz):
    """
    Compute pseudorapidity (eta) from 3-momentum components.

    Parameters:
        px, py, pz : float or array-like

    Returns:
        eta : float or numpy array
    """
    px = np.asarray(px)
    py = np.asarray(py)
    pz = np.asarray(pz)

    pT = np.sqrt(px**2 + py**2)

    # Use asinh form for numerical stability
    eta = np.arcsinh(pz / pT)

    return eta


In [6]:
import time
from time import perf_counter


def noise_probability_gaussian(freq_kHz=100,time_window_ns=10):
    p = freq_kHz*1000 *time_window_ns/1000000000
    sigma_p = p * 0.01
    single_ph_emission_p = np.random.normal(0, 1) * sigma_p + p
    if single_ph_emission_p < 0:
        single_ph_emission_p = 0

    return single_ph_emission_p

def noise_generation(df_str="../csv/cellmap.csv", freq_kHz=100, time_window_ns=10, event=99999):
    df = pd.read_csv(df_str,  sep = ',')
    hitlist = []
    for index, row in df.iterrows():
        hit_prob = noise_probability_gaussian(freq_kHz,time_window_ns)
        limit = np.random.uniform(0,1)
        #print("p=%f, limit=%f", hit_prob,limit)
        if hit_prob > limit:
            row = row.copy()
            row["event"] = event
            row["time"] = 1
            row["charge"] = 280
            row["pindex"] = 0
            hitlist.append(row)
                        #print(d4)
    return pd.DataFrame(hitlist)

def noiseOnly_dataset(nevents=10, freq_kHz=100, time_window_ns=10):
    start = time.perf_counter()
    df_all = []
    for ev in range(nevents):
        print(f"Evento di rumore #{ev}")
        df_new = noise_generation("../csv/cellmap.csv", freq_kHz, time_window_ns, event=ev)
        if not df_new.empty:
            df_all.append(df_new)

    if df_all:
        df_final = pd.concat(df_all, ignore_index=True)
        df_final = df_final[['event', 'cell_id', 'sector', 'time', 'charge', 'pindex',
                                  'gx', 'gy', 'gz', 'lx', 'ly', 'pdu', 'sipm', 'xi', 'yi']]
    else:
        df_final = pd.DataFrame()  # Nessun evento generato



    end = time.perf_counter()

    print(f"Tempo totale {end - start:.4f} secondi ==> {(end - start)/nevents:.4f}")

    return df_final


def noise_generation_optimized(cellmap, freq_kHz=100, time_window_ns=10, event=99999):
    p = freq_kHz * 1000 * time_window_ns / 1_000_000_000
    sigma_p = p * 0.01

    size = len(cellmap)

    # Genera probabilità gaussiane per tutte le righe
    emission_probs = np.random.normal(loc=p, scale=sigma_p, size=size)

    # Genera probabilità radiali
    #radii = np.sqrt(cellmap["gx"]*cellmap["gx"]+cellmap["gy"]*cellmap["gy"])
    #emission_probs = noise_probability_radial(luminosity_fbi=freq_kHz,time_window_ns=time_window_ns,radius=radii)

    emission_probs = np.clip(emission_probs, 0, None)

    # Genera limiti casuali uniformi
    thresholds = np.random.uniform(0, 1, size)

    # Filtra solo i "veri hit"
    mask = emission_probs > thresholds
    hits = cellmap[mask].copy()

    # Aggiungi colonne
    hits["event"] = event
    hits["time"] = np.random.uniform(0,10000) #added timing
    hits["charge"] = 280
    hits["pindex"] = 0

    return hits


def noiseOnly_dataset_optimized(cellmap, nevents=10, freq_kHz=100, time_window_ns=10):
    start = time.perf_counter()
    all_hits = []

    for ev in range(nevents):
        hits = noise_generation_optimized(cellmap, freq_kHz, time_window_ns, event=ev)
        all_hits.append(hits)

    df_final = pd.concat(all_hits, ignore_index=True)
    df_final = df_final[['event', 'cell_id', 'sector', 'time', 'charge', 'pindex',
                                  'gx', 'gy', 'gz', 'lx', 'ly', 'pdu', 'sipm', 'xi', 'yi']]
    end = time.perf_counter()

    print(f"Tempo totale: {end - start:.4f}s → {(end - start)/nevents:.4f}s per evento")
    return df_final



In [7]:
import time

def grid_ALCOR_optimized(size, df):
    grid_matrix = np.zeros((1000, 6, 5, size), dtype=int)

    # Filtra il DataFrame per eventi, settori, subsezioni e pdu validi
    df = df[df["event"] < 1000]
    df = df[df["sector"] < 6]
    df["subsec"] = df["pdu"] // 42
    df = df[df["subsec"] < 5]
    df["subpdu"] = df["pdu"] % 42

    # Raggruppa per indici rilevanti
    grouped = df.groupby(["event", "sector", "subsec", "subpdu"]).size()

    # Assegna i conteggi alla matrice
    for (ev, sec, subsec, subpdu), count in grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] = count

    return grid_matrix

def grid_ALCOR_optimized_timingaware(size, df, binwidth):
    grid_matrix = np.zeros((1000, 6, 5, size, int(ten/binwidth)), dtype=int)

    # Filtra il DataFrame per eventi, settori, subsezioni e pdu validi
    df = df[df["event"] < 1000]
    df = df[df["sector"] < 6]
    df["subsec"] = df["pdu"] // 42
    df = df[df["subsec"] < 5]
    df["subpdu"] = df["pdu"] % 42
    df["time"] = df["time"].astype(np.float32)
    timing_col = pd.DataFrame({'time': np.random.uniform(0, ten, len(df["time"]))}, dtype=np.float32) #badile
    df.update(timing_col)
    df["coarse_time"] = df["time"] // binwidth # + 1 # badile
    #df["coarse_time"] = (df["time"] // binwidth) #.astype(int)
    # Raggruppa per indici rilevanti
    grouped = df.groupby(["event", "sector", "subsec", "subpdu","coarse_time"]).size()

    #Trova il coarse_time più frequente per ogni cella
    #idx = grouped.groupby(level=[0,1,2,3]).idxmax()

    # Assegna i TEMPI alla matrice
    for (ev, sec, subsec, subpdu, coarse_time), count in grouped.items():
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)] = count #* 2.54
        #print(coarse_time)
        #grid_matrix[ev, sec, subsec, subpdu] =  np.bincount(coarse_time).argmax() * 2.54
    return grid_matrix


def grid_ALCOR_optimized_signalonly(size, signal_frame):
    grid_matrix = np.zeros((1000, 6, 5, size), dtype=int)

    # Filtra il DataFrame per eventi, settori, subsezioni e ogni 42 pdu
    sf = signal_frame[signal_frame["event"] < 1000]
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    #print(sf)

    # Raggruppa per indici rilevanti
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    # Assegna i conteggi alla matrice (segnale)
    for (ev, sec, subsec, subpdu), count in signal_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] = count

    return grid_matrix




def grid_ALCOR_optimized_signalplusnoise(size, noise_frame, signal_frame):
    grid_matrix = np.zeros((1000, 6, 5, size), dtype=int)

    # Filtra il DataFrame per eventi, settori, subsezioni e ogni 42 pdu
    nf = noise_frame[noise_frame["event"] < 1000]
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42

    sf = signal_frame[signal_frame["event"] < 1000]
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    #print(sf)

    # Raggruppa per indici rilevanti
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    # Assegna i conteggi alla matrice (segnale)
    for (ev, sec, subsec, subpdu), count in signal_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] = count



    # Colonne da confrontare
    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm",  "xi", "yi"]

    # 1. Subset dei DataFrame
    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    # 2. Converti le righe di signal in un set di tuple per confronto efficiente
    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    # 3. Crea una maschera: True se la riga di noise NON è in signal
    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)

    # 4. Applica la maschera a noise per ottenere il risultato filtrato
    filtered_nf = nf[mask]

    # Raggruppa per indici rilevanti
    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu"]).size()



    # Assegna i conteggi alla matrice (noise SE non c'è già segnale)
    for (ev, sec, subsec, subpdu), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] += count

    return grid_matrix


def grid_ALCOR_optimized_signalplusnoise_NOZEROS(size, noise_frame, signal_frame):

    grid_matrix = np.zeros((1000, 6, 5, size), dtype=int)

    # --- Preprocessing NOISE ---
    nf = noise_frame[noise_frame["event"] < 1000].copy()
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42

    # --- Preprocessing SIGNAL ---
    sf = signal_frame[signal_frame["event"] < 1000].copy()
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    # --- ⛔ Mantieni solo gli eventi che hanno segnale ---
    eventi_con_segnale = sf["event"].unique()
    nf = nf[nf["event"].isin(eventi_con_segnale)]

    # --- Signal grouping ---
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    for (ev, sec, subsec, subpdu), count in signal_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] = count

    # --- Noise minus signal (stessa posizione SiPM) ---

    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm", "xi", "yi"]

    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)
    filtered_nf = nf[mask]

    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    for (ev, sec, subsec, subpdu), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] += count

    # --- ⛔ Mantieni solo gli eventi che hanno segnale --- ===> PADDING (solved, la rete lo impara da sola)
    #ev_finali = np.sort(eventi_con_segnale)
    #grid_matrix = grid_matrix[ev_finali]

    return grid_matrix


def grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware(size, noise_frame, signal_frame, binwidth):

    grid_matrix = np.zeros((1000, 6, 5, size, int(ten/binwidth)), dtype=int)

    # --- Preprocessing NOISE ---
    nf = noise_frame[noise_frame["event"] < 1000].copy()
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42
    nf["time"] = nf["time"].astype(np.float32)
    timing_col = pd.DataFrame({'time': np.random.uniform(0, ten, len(nf["time"]))}, dtype=np.float32)
    nf.update(timing_col)
    nf["coarse_time"] = (nf["time"] // binwidth)# + 1

    # --- Preprocessing SIGNAL ---

    for ev in range(max(signal_frame["event"])):
        ev=ev+1
        #print(ev)
        #start = perf_counter()
        df = signal_frame[signal_frame["event"]==ev]
        min_time = min(df["time"])
        #time_col = ((df["time"].astype(np.float32)-min_time)/16)%10.32 #badile era fratto /16
        time_col = ((df["time"].astype(np.float32)-min_time))%10.32
        #time_col = (df["time"]/16)%20.32
        if (ev==1):
            sig_times = time_col
        else:
            df_previous = sig_times
            sig_times =  pd.concat([df_previous.astype(np.float32), time_col.astype(np.float32)])
        end = perf_counter()
        #if(ev%100 == 0):
            #print(f"Tempo totale per eventp: {end - start:.4f}s")

    sf = signal_frame[signal_frame["event"] < 1000].copy()
    #sf = sf[sf["time"] < 10000]
    sf["time"] = sf["time"].astype(np.float32)
    sf.update(sig_times)
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42


    #sf["coarse_time"] = (((sf["time"]*16)-((sf["time"]*16)//10.16)*10.16) // binwidth)
    sf["coarse_time"] = sf["time"] // binwidth

    # --- ⛔ Mantieni solo gli eventi che hanno segnale ---
    eventi_con_segnale = sf["event"].unique()
    nf = nf[nf["event"].isin(eventi_con_segnale)]

    # --- Signal grouping ---
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu","coarse_time"]).size()

    # --- Noise minus signal (stessa posizione SiPM) ---

    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm", "xi", "yi"]

    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)
    filtered_nf = nf[mask]

    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu", "coarse_time"]).size()
    for (ev, sec, subsec, subpdu, coarse_time), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%nbins] = count #* 2.54

    #all_grouped = pd.concat([signal_grouped, noise_grouped], axis=0)
    #idx = all_grouped.groupby(level=[0,1,2,3]).idxmax()

    #for (ev, sec, subsec, subpdu, coarse_time), count in all_grouped.items():
        #grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] = count #* 2.54
    for (ev, sec, subsec, subpdu, coarse_time), count in signal_grouped.items():
        #min_df = sf[sf["event"]==ev]
        #min_time = min(min_df["time"])
        #time_col = ((df["time"]-min_time)/16)%20.32
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%nbins] += count #* 2.54

    # --- ⛔ Mantieni solo gli eventi che hanno segnale --- ===> PADDING (solved, la rete lo impara da sola)
    #ev_finali = np.sort(eventi_con_segnale)
    #grid_matrix = grid_matrix[ev_finali]

    return grid_matrix

In [8]:
def grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware_acceptancefiltered(size, noise_frame, rec_signal_frame, sim_signal_frame, binwidth):

    grid_matrix = np.zeros((1000, 6, 5, size, int(ten/binwidth)), dtype=int)

    # --- Preprocessing NOISE ---
    nf = noise_frame[noise_frame["event"] < 1000].copy()
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42
    nf["time"] = nf["time"].astype(np.float32)
    timing_col = pd.DataFrame({'time': np.random.uniform(0, ten, len(nf["time"]))}, dtype=np.float32)
    nf.update(timing_col)
    nf["coarse_time"] = (nf["time"] // binwidth)# + 1

    # --- Preprocessing SIGNAL ---

    for ev in range(max(rec_signal_frame["event"])):
        ev=ev+1
        #print(ev)
        #start = perf_counter()
        df = rec_signal_frame[rec_signal_frame["event"]==ev]
        min_time = min(df["time"])
        time_col = ((df["time"].astype(np.float32)-min_time)/16)%10.32
        #time_col = (df["time"]/16)%20.32
        if (ev==1):
            sig_times = time_col
        else:
            df_previous = sig_times
            sig_times =  pd.concat([df_previous.astype(np.float32), time_col.astype(np.float32)])
        end = perf_counter()
        #if(ev%100 == 0):
            #print(f"Tempo totale per eventp: {end - start:.4f}s")

    sf = rec_signal_frame[rec_signal_frame["event"] < 1000].copy()
    #sf = sf[sf["time"] < 10000]
    sf["time"] = sf["time"].astype(np.float32)
    sf.update(sig_times)
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    #acceptance filter
    sim_sf = sim_signal_frame[sim_signal_frame["event"] < 1000].copy()
    eta = pseudorapidity(sim_sf["px"], sim_sf["py"], sim_sf["pz"])
    eta_mask = (eta >= 1.5) & (eta<=3.5)
    sim_sf = sim_sf[eta_mask]
    #sf = sf[sf["pindex"].isin(sim_sf["pindex"])]
    sf = sf.merge(sim_sf[["event", "pindex"]], on=["event", "pindex"],how="inner")

    #sf["coarse_time"] = (((sf["time"]*16)-((sf["time"]*16)//10.16)*10.16) // binwidth)
    sf["coarse_time"] = sf["time"] // binwidth

    # --- ⛔ Mantieni solo gli eventi che hanno segnale ---
    eventi_con_segnale = sf["event"].unique()
    nf = nf[nf["event"].isin(eventi_con_segnale)]

    # --- Signal grouping ---
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu","coarse_time"]).size()

    # --- Noise minus signal (stessa posizione SiPM) ---

    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm", "xi", "yi"]

    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)
    filtered_nf = nf[mask]

    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu", "coarse_time"]).size()
    for (ev, sec, subsec, subpdu, coarse_time), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] = count #* 2.54

    #all_grouped = pd.concat([signal_grouped, noise_grouped], axis=0)
    #idx = all_grouped.groupby(level=[0,1,2,3]).idxmax()

    #for (ev, sec, subsec, subpdu, coarse_time), count in all_grouped.items():
        #grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] = count #* 2.54
    for (ev, sec, subsec, subpdu, coarse_time), count in signal_grouped.items():
        #min_df = sf[sf["event"]==ev]
        #min_time = min(min_df["time"])
        #time_col = ((df["time"]-min_time)/16)%20.32
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] += count #* 2.54

    # --- ⛔ Mantieni solo gli eventi che hanno segnale --- ===> PADDING (solved, la rete lo impara da sola)
    #ev_finali = np.sort(eventi_con_segnale)
    #grid_matrix = grid_matrix[ev_finali]

    return grid_matrix


def grid_ALCOR_optimized_signalplusnoise_NOZEROS_acceptancefiltered(size, noise_frame, rec_signal_frame,sim_signal_frame):

    grid_matrix = np.zeros((1000, 6, 5, size), dtype=int)

    # --- Preprocessing NOISE ---
    nf = noise_frame[noise_frame["event"] < 1000].copy()
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42

    # --- Preprocessing SIGNAL ---
    sf = rec_signal_frame[rec_signal_frame["event"] < 1000].copy()
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    #acceptance filter
    sim_sf = sim_signal_frame[sim_signal_frame["event"] < 1000].copy()
    eta = pseudorapidity(sim_sf["px"], sim_sf["py"], sim_sf["pz"])
    eta_mask = (eta >= 1.5) & (eta<=3.5)
    sim_sf = sim_sf[eta_mask]
    #sf = sf[sf["pindex"].isin(sim_sf["pindex"])]
    sf = sf.merge(sim_sf[["event", "pindex"]], on=["event", "pindex"],how="inner")

    # --- ⛔ Mantieni solo gli eventi che hanno segnale ---
    eventi_con_segnale = sf["event"].unique()
    nf = nf[nf["event"].isin(eventi_con_segnale)]

    # --- Signal grouping ---
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    for (ev, sec, subsec, subpdu), count in signal_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] = count

    # --- Noise minus signal (stessa posizione SiPM) ---

    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm", "xi", "yi"]

    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)
    filtered_nf = nf[mask]

    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    for (ev, sec, subsec, subpdu), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] += count

    # --- ⛔ Mantieni solo gli eventi che hanno segnale --- ===> PADDING (solved, la rete lo impara da sola)
    #ev_finali = np.sort(eventi_con_segnale)
    #grid_matrix = grid_matrix[ev_finali]

    return grid_matrix

In [9]:
def grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware_nparticles(size, noise_frame, rec_signal_frame, sim_signal_frame, binwidth):

    grid_matrix = np.zeros((1000, 6, 5, size, int(ten/binwidth)), dtype=int)

    # --- Preprocessing NOISE ---
    nf = noise_frame[noise_frame["event"] < 1000].copy()
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42
    nf["time"] = nf["time"].astype(np.float32)
    timing_col = pd.DataFrame({'time': np.random.uniform(0, ten, len(nf["time"]))}, dtype=np.float32)
    nf.update(timing_col)
    nf["coarse_time"] = (nf["time"] // binwidth)# + 1

    # --- Preprocessing SIGNAL ---

    for ev in range(max(rec_signal_frame["event"])):
        ev=ev+1
        #print(ev)
        #start = perf_counter()
        df = rec_signal_frame[rec_signal_frame["event"]==ev]
        min_time = min(df["time"])
        time_col = ((df["time"].astype(np.float32)-min_time)/16)%10.32
        #time_col = (df["time"]/16)%20.32
        if (ev==1):
            sig_times = time_col
        else:
            df_previous = sig_times
            sig_times =  pd.concat([df_previous.astype(np.float32), time_col.astype(np.float32)])
        end = perf_counter()
        #if(ev%100 == 0):
            #print(f"Tempo totale per eventp: {end - start:.4f}s")

    sf = rec_signal_frame[rec_signal_frame["event"] < 1000].copy()
    #sf = sf[sf["time"] < 10000]
    sf["time"] = sf["time"].astype(np.float32)
    sf.update(sig_times)
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    #nparticles
    sim_sf = sim_signal_frame[sim_signal_frame["event"] < 1000].copy()
    nparticles_count = sim_sf.groupby("event")["pindex"].nunique().clip(upper=3)
    nparticles_capped = nparticles_count.values
    y_onehot = pd.get_dummies(nparticles_capped)
    y_onehot = y_onehot.reindex(columns=[0, 1, 2, 3], fill_value=0).astype(int)
    y_onehot = y_onehot.to_numpy().astype(int)

    #sf["coarse_time"] = (((sf["time"]*16)-((sf["time"]*16)//10.16)*10.16) // binwidth)
    sf["coarse_time"] = sf["time"] // binwidth

    # --- ⛔ Mantieni solo gli eventi che hanno segnale ---
    eventi_con_segnale = sf["event"].unique()
    nf = nf[nf["event"].isin(eventi_con_segnale)]

    # --- Signal grouping ---
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu","coarse_time"]).size()

    # --- Noise minus signal (stessa posizione SiPM) ---

    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm", "xi", "yi"]

    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)
    filtered_nf = nf[mask]

    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu", "coarse_time"]).size()
    for (ev, sec, subsec, subpdu, coarse_time), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] = count #* 2.54

    #all_grouped = pd.concat([signal_grouped, noise_grouped], axis=0)
    #idx = all_grouped.groupby(level=[0,1,2,3]).idxmax()

    #for (ev, sec, subsec, subpdu, coarse_time), count in all_grouped.items():
        #grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] = count #* 2.54
    for (ev, sec, subsec, subpdu, coarse_time), count in signal_grouped.items():
        #min_df = sf[sf["event"]==ev]
        #min_time = min(min_df["time"])
        #time_col = ((df["time"]-min_time)/16)%20.32
        grid_matrix[ev, sec, subsec, subpdu, int(coarse_time)%4] += count #* 2.54

    # --- ⛔ Mantieni solo gli eventi che hanno segnale --- ===> PADDING (solved, la rete lo impara da sola)
    #ev_finali = np.sort(eventi_con_segnale)
    #grid_matrix = grid_matrix[ev_finali]

    return grid_matrix, y_onehot


def grid_ALCOR_optimized_signalplusnoise_NOZEROS_nparticles(size, noise_frame, rec_signal_frame,sim_signal_frame):

    #grid_matrix = np.zeros((1000, 6, 5, size), dtype=int)

    # --- Preprocessing NOISE ---
    nf = noise_frame[noise_frame["event"] < 1000].copy()
    nf = nf[nf["sector"] < 6]
    nf["subsec"] = nf["pdu"] // 42
    nf = nf[nf["subsec"] < 5]
    nf["subpdu"] = nf["pdu"] % 42

    # --- Preprocessing SIGNAL ---
    sf = rec_signal_frame[rec_signal_frame["event"] < 1000].copy()
    sf = sf[sf["pindex"] > 0]
    sf = sf[sf["sector"] < 6]
    sf["subsec"] = sf["pdu"] // 42
    sf = sf[sf["subsec"] < 5]
    sf["subpdu"] = sf["pdu"] % 42

    grid_matrix = np.zeros((len(sf["event"].unique()), 6, 5, size), dtype=int)
    nparticles_y_array = np.zeros((len(sf["event"].unique()),4), dtype=int)


    #nparticles
    sim_sf = sim_signal_frame[sim_signal_frame["event"] < 1000].copy()
    print(len(sim_sf["event"].unique()))
    nparticles_count = sim_sf.groupby("event")["pindex"].nunique().clip(upper=3)
    nparticles_capped = nparticles_count.values
    y_onehot = pd.get_dummies(nparticles_capped)
    y_onehot = y_onehot.reindex(columns=[0, 1, 2, 3], fill_value=0).astype(int)
    y_onehot = y_onehot.to_numpy().astype(int)

    # --- ⛔ Mantieni solo gli eventi che hanno segnale ---
    eventi_con_segnale = sf["event"].unique()
    nf = nf[nf["event"].isin(eventi_con_segnale)]

    # --- Signal grouping ---
    signal_grouped = sf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    for (ev, sec, subsec, subpdu), count in signal_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] = count

    # --- Noise minus signal (stessa posizione SiPM) ---

    colonne_da_confrontare = ["event", "sector", "subsec", "subpdu", "sipm", "xi", "yi"]

    signal_subset = sf[colonne_da_confrontare]
    noise_subset = nf[colonne_da_confrontare]

    signal_tuples = set(map(tuple, signal_subset.to_numpy()))

    mask = noise_subset.apply(lambda row: tuple(row) not in signal_tuples, axis=1)
    filtered_nf = nf[mask]

    noise_grouped = filtered_nf.groupby(["event", "sector", "subsec", "subpdu"]).size()

    for (ev, sec, subsec, subpdu), count in noise_grouped.items():
        grid_matrix[ev, sec, subsec, subpdu] += count

    # --- ⛔ Mantieni solo gli eventi che hanno segnale --- ===> PADDING (solved, la rete lo impara da sola)
    #ev_finali = np.sort(eventi_con_segnale)
    #grid_matrix = grid_matrix[ev_finali]

    return grid_matrix, y_onehot

In [10]:
n_reco_files = 100
nevents = 1000

#cellmap = pd.read_csv("/tmp/cellmap.csv",  sep = ',')
cellmap = pd.read_csv("/content/drive/MyDrive/project_files/cellmap.csv", sep=',')



FREQ = 150
frequences = [FREQ]
GEN =   False

In [11]:
if(GEN):
    for f in frequences:
        #print(f)
        for nfile in range(n_reco_files):
            print(f"\n--- Generazione file ricostruito {nfile} ==> noise rate = {f} kHz ---")
            df_noise = noiseOnly_dataset_optimized(cellmap,nevents=nevents, freq_kHz=f,time_window_ns=10)
            #time info added
            df_noise["time"] = df_noise["time"].astype(np.float32)
            timing_col = pd.DataFrame({'time': np.random.uniform(0, ten, len(df_noise["time"]))}, dtype=np.float32)
            df_noise.update(timing_col)
              # Definiamo la cartella Google Drive
            cartella_destinazione = "/content/drive/MyDrive/project_files/Noise_files/with_time/csv_reco/"
            # Questo comando crea la cartella su Drive se non esiste ancora
            os.makedirs(cartella_destinazione, exist_ok=True)

            output_filename = os.path.join(cartella_destinazione, f"noise_reco_{nfile}_{f}kHz_10ns.csv")
            df_noise.to_csv(output_filename, index=False)
            print(f"Salvato in {output_filename}")

In [12]:
import os
import pandas as pd
import numpy as np
import time

nfiles = 582
df_rec_signal_list = []
df_index = 0
new_file_init = True

start = time.perf_counter()
print(f"RECONSTRUCTION DATASET")
for f in range(nfiles):
    print(f"SIGNAL FILE n.{f}")

    file_string = f"/content/drive/MyDrive/project_files/Bkg_new/rec_hits_{f+1:04d}.csv"

    if not os.path.exists(file_string):
        print(file_string + " skipped!")
        continue

    # Leggi CSV
    df_rec_signal = pd.read_csv(file_string, sep=',')

    # Salta DataFrame vuoti o senza colonna "event"
    if df_rec_signal.empty or "event" not in df_rec_signal.columns:
        continue

    # Riallinea eventi locali partendo da 0
    df_rec_signal["event"] = df_rec_signal.groupby("event", sort=False).ngroup()

    # Converte a intero per sicurezza
    df_rec_signal["event"] = df_rec_signal["event"].astype(int)

    nevents = df_rec_signal["event"].max()

    if new_file_init:
        # Split se supera 999
        if nevents > 999:
            df1 = df_rec_signal[df_rec_signal["event"] <= 999].copy()
            df2 = df_rec_signal[df_rec_signal["event"] > 999].copy()

            if not df2.empty:
                df2["event"] = (df2["event"] - df2["event"].min()) % 1000
                df2["event"] = df2["event"].astype(int)

            df_rec_signal_list.append(df1)
            df_index += 1
            df_rec_signal_list.append(df2)
        else:
            df_rec_signal_list.append(df_rec_signal)
        new_file_init = False
    else:
        # Offset globale rispetto al precedente, modulo 1000 per wrap
        df_previous = df_rec_signal_list[df_index]
        prev_max = df_previous["event"].max() if not df_previous.empty else -1
        df_rec_signal["event"] = (df_rec_signal["event"] + prev_max + 1)# % 1000
        df_rec_signal["event"] = df_rec_signal["event"].astype(int)

        # Concatenate temporanea
        df_conc = pd.concat([df_previous, df_rec_signal], ignore_index=True)
        nev = df_conc["event"].max()

        if nev > 999:
            df1 = df_conc[df_conc["event"] <= 999].copy()
            df2 = df_conc[df_conc["event"] > 999].copy()

            if not df2.empty:
                df2["event"] = (df2["event"] - df2["event"].min()) % 1000
                df2["event"] = df2["event"].astype(int)

            df_rec_signal_list[df_index] = df1
            df_index += 1
            df_rec_signal_list.append(df2)
            new_file_init = False
        else:
            df_rec_signal_list[df_index] = df_conc
            if nev == 999:
                new_file_init = True

end = time.perf_counter()
print(f"Tempo totale: {end - start:.4f}s")


RECONSTRUCTION DATASET
SIGNAL FILE n.0
SIGNAL FILE n.1
SIGNAL FILE n.2
SIGNAL FILE n.3
SIGNAL FILE n.4
/content/drive/MyDrive/project_files/Bkg_new/rec_hits_0005.csv skipped!
SIGNAL FILE n.5
SIGNAL FILE n.6
SIGNAL FILE n.7
SIGNAL FILE n.8
SIGNAL FILE n.9
/content/drive/MyDrive/project_files/Bkg_new/rec_hits_0010.csv skipped!
SIGNAL FILE n.10
SIGNAL FILE n.11
SIGNAL FILE n.12
SIGNAL FILE n.13
SIGNAL FILE n.14
SIGNAL FILE n.15
/content/drive/MyDrive/project_files/Bkg_new/rec_hits_0016.csv skipped!
SIGNAL FILE n.16
SIGNAL FILE n.17
SIGNAL FILE n.18
/content/drive/MyDrive/project_files/Bkg_new/rec_hits_0019.csv skipped!
SIGNAL FILE n.19
SIGNAL FILE n.20
SIGNAL FILE n.21
SIGNAL FILE n.22
SIGNAL FILE n.23
/content/drive/MyDrive/project_files/Bkg_new/rec_hits_0024.csv skipped!
SIGNAL FILE n.24
SIGNAL FILE n.25
SIGNAL FILE n.26
/content/drive/MyDrive/project_files/Bkg_new/rec_hits_0027.csv skipped!
SIGNAL FILE n.27
SIGNAL FILE n.28
SIGNAL FILE n.29
SIGNAL FILE n.30
SIGNAL FILE n.31
SIGNAL FILE

In [13]:
import os
import pandas as pd
import numpy as np
import time

nfiles = 582
df_sim_signal_list = []
df_index = 0
new_file_init = True

start = time.perf_counter()
print(f"SIMULATION DATASET")
for f in range(nfiles):
    print(f"SIGNAL FILE n.{f}")

    file_string = f"Bkg_1SignalPer2usframe_2601/sim_parts_{f+1:04d}.csv"

    if not os.path.exists(file_string):
        print(file_string + " skipped!")
        continue

    # Leggi CSV
    df_sim_signal = pd.read_csv(file_string, sep=',')

    # Salta DataFrame vuoti o senza colonna "event"
    if df_sim_signal.empty or "event" not in df_sim_signal.columns:
        continue

    # Riallinea eventi locali partendo da 0
    df_sim_signal["event"] = df_sim_signal.groupby("event", sort=False).ngroup()

    # Converte a intero per sicurezza
    df_sim_signal["event"] = df_sim_signal["event"].astype(int)

    nevents = df_sim_signal["event"].max()

    if new_file_init:
        # Split se supera 999
        if nevents > 999:
            df1 = df_sim_signal[df_sim_signal["event"] <= 999].copy()
            df2 = df_sim_signal[df_sim_signal["event"] > 999].copy()

            if not df2.empty:
                df2["event"] = (df2["event"] - df2["event"].min()) % 1000
                df2["event"] = df2["event"].astype(int)

            df_sim_signal_list.append(df1)
            df_index += 1
            df_sim_signal_list.append(df2)
        else:
            df_sim_signal_list.append(df_rec_signal)
        new_file_init = False
    else:
        # Offset globale rispetto al precedente, modulo 1000 per wrap
        df_previous = df_sim_signal_list[df_index]
        prev_max = df_previous["event"].max() if not df_previous.empty else -1
        df_sim_signal["event"] = (df_sim_signal["event"] + prev_max + 1)# % 1000
        df_sim_signal["event"] = df_sim_signal["event"].astype(int)

        # Concatenate temporanea
        df_conc = pd.concat([df_previous, df_sim_signal], ignore_index=True)
        nev = df_conc["event"].max()

        if nev > 999:
            df1 = df_conc[df_conc["event"] <= 999].copy()
            df2 = df_conc[df_conc["event"] > 999].copy()

            if not df2.empty:
                df2["event"] = (df2["event"] - df2["event"].min()) % 1000
                df2["event"] = df2["event"].astype(int)

            df_sim_signal_list[df_index] = df1
            df_index += 1
            df_sim_signal_list.append(df2)
            new_file_init = False
        else:
            df_sim_signal_list[df_index] = df_conc
            if nev == 999:
                new_file_init = True

end = time.perf_counter()
print(f"Tempo totale merging SIG+NOISE: {end - start:.4f}s")

SIMULATION DATASET
SIGNAL FILE n.0
Bkg_1SignalPer2usframe_2601/sim_parts_0001.csv skipped!
SIGNAL FILE n.1
Bkg_1SignalPer2usframe_2601/sim_parts_0002.csv skipped!
SIGNAL FILE n.2
Bkg_1SignalPer2usframe_2601/sim_parts_0003.csv skipped!
SIGNAL FILE n.3
Bkg_1SignalPer2usframe_2601/sim_parts_0004.csv skipped!
SIGNAL FILE n.4
Bkg_1SignalPer2usframe_2601/sim_parts_0005.csv skipped!
SIGNAL FILE n.5
Bkg_1SignalPer2usframe_2601/sim_parts_0006.csv skipped!
SIGNAL FILE n.6
Bkg_1SignalPer2usframe_2601/sim_parts_0007.csv skipped!
SIGNAL FILE n.7
Bkg_1SignalPer2usframe_2601/sim_parts_0008.csv skipped!
SIGNAL FILE n.8
Bkg_1SignalPer2usframe_2601/sim_parts_0009.csv skipped!
SIGNAL FILE n.9
Bkg_1SignalPer2usframe_2601/sim_parts_0010.csv skipped!
SIGNAL FILE n.10
Bkg_1SignalPer2usframe_2601/sim_parts_0011.csv skipped!
SIGNAL FILE n.11
Bkg_1SignalPer2usframe_2601/sim_parts_0012.csv skipped!
SIGNAL FILE n.12
Bkg_1SignalPer2usframe_2601/sim_parts_0013.csv skipped!
SIGNAL FILE n.13
Bkg_1SignalPer2usframe_26

In [14]:
input_signal_matrices = []
input_nparticles_array = []


for f in range(1,len(df_rec_signal_list)-1):
    print("SIGNAL FILE n.%d" % (f))
    start = time.perf_counter()
    #df_rec_signal     = pd.read_csv("/apotto/home1/aliens/apepic/work/rossi/drich_eic/csv/no_noise_events_Bkg_realistic_25_05/rec_hits.csv.%04d.csv" % (f+1), sep = ',')
    df_rec_signal    = df_rec_signal_list[f]
    #df_sim_signal    = df_sim_signal_list[f]
    df_rec_noise     = pd.read_csv(f"/content/drive/MyDrive/project_files/Noise_files/with_time/csv_reco/noise_reco_{f}_{FREQ}kHz_10ns.csv")
    #ALCOR_gridmatrix_total, nparticles_array = grid_ALCOR_optimized_signalplusnoise_NOZEROS_nparticles(42, df_rec_noise, df_rec_signal, df_sim_signal)
    #ALCOR_gridmatrix_total, nparticles_array = grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware_nparticles(42, df_rec_noise, df_rec_signal, df_sim_signal, binwidth,)
    #ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS_acceptancefiltered(42, df_rec_noise, df_rec_signal, df_sim_signal)
    #ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware_acceptancefiltered(42, df_rec_noise, df_rec_signal, df_sim_signal, binwidth,)
    ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware(42, df_rec_noise, df_rec_signal,binwidth)
    #ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS(42, df_rec_noise, df_rec_signal)
    print(ALCOR_gridmatrix_total.shape)
    #print(nparticles_array.shape)
    ALCOR_gridmatrix_total = ALCOR_gridmatrix_total.reshape(ALCOR_gridmatrix_total.shape[0],6,5,42*int(ten/binwidth))
    #ALCOR_gridmatrix_total = ALCOR_gridmatrix_total.reshape(ALCOR_gridmatrix_total.shape[0],6,5,42)

    input_signal_matrices.append(ALCOR_gridmatrix_total)

    #input_nparticles_array.append(nparticles_array)


    end = time.perf_counter()
    print(f"Tempo totale: {end - start:.4f}s")


SIGNAL FILE n.1
(1000, 6, 5, 42, 10)
Tempo totale: 7.4984s
SIGNAL FILE n.2
(1000, 6, 5, 42, 10)
Tempo totale: 5.1654s
SIGNAL FILE n.3
(1000, 6, 5, 42, 10)
Tempo totale: 7.0314s
SIGNAL FILE n.4
(1000, 6, 5, 42, 10)
Tempo totale: 5.5124s
SIGNAL FILE n.5
(1000, 6, 5, 42, 10)
Tempo totale: 5.7381s
SIGNAL FILE n.6
(1000, 6, 5, 42, 10)
Tempo totale: 6.1179s
SIGNAL FILE n.7
(1000, 6, 5, 42, 10)
Tempo totale: 5.4705s
SIGNAL FILE n.8
(1000, 6, 5, 42, 10)
Tempo totale: 7.0112s
SIGNAL FILE n.9
(1000, 6, 5, 42, 10)
Tempo totale: 5.5654s
SIGNAL FILE n.10
(1000, 6, 5, 42, 10)
Tempo totale: 7.2892s
SIGNAL FILE n.11
(1000, 6, 5, 42, 10)
Tempo totale: 5.3728s
SIGNAL FILE n.12
(1000, 6, 5, 42, 10)
Tempo totale: 5.8772s
SIGNAL FILE n.13
(1000, 6, 5, 42, 10)
Tempo totale: 6.0772s
SIGNAL FILE n.14
(1000, 6, 5, 42, 10)
Tempo totale: 5.3653s
SIGNAL FILE n.15
(1000, 6, 5, 42, 10)
Tempo totale: 7.0110s
SIGNAL FILE n.16
(1000, 6, 5, 42, 10)
Tempo totale: 5.4181s
SIGNAL FILE n.17
(1000, 6, 5, 42, 10)
Tempo total

In [15]:
nevents=1000
if(GEN):
    for f in frequences:
        for nfile in range(n_reco_files):
            print(f"\n--- Generazione file ricostruito {nfile} ==> noise rate = {f} kHz ---")
            df_noise = noiseOnly_dataset_optimized(cellmap,nevents=nevents, freq_kHz=f,time_window_ns=10)

            #time info added
            df_noise["time"] = df_noise["time"].astype(np.float32)
            timing_col = pd.DataFrame({'time': np.random.uniform(0, ten, len(df_noise["time"]))}, dtype=np.float32)
            df_noise.update(timing_col)

            output_filename = f"/content/drive/MyDrive/project_files/Noise_files/with_time/csv_reco/only_noise_reco_{nfile}_{f}kHz_10ns.csv"
            df_noise.to_csv(output_filename, index=False)
            print(f"Salvato in {output_filename}")

In [16]:
#input_signal_matrices = []
#input_nparticles_array = []
#
#
#for f in range(1,num_signal_ROOT_files-1):
#    print("SIGNAL FILE n.%d" % (f))
#    start = time.perf_counter()
#    #df_rec_signal     = pd.read_csv("/apotto/home1/aliens/apepic/work/rossi/drich_eic/csv/no_noise_events_Bkg_realistic_25_05/rec_hits.csv.%04d.csv" % (f+1), sep = ',')
#    df_rec_signal    = df_rec_signal_list[f]
#    df_sim_signal    = df_sim_signal_list[f]
#    df_rec_noise     = pd.read_csv(f"csv_reco/noise_reco_{f}_{FREQ}kHz_10ns.csv")
#
#    #ALCOR_gridmatrix_total, nparticles_array = grid_ALCOR_optimized_signalplusnoise_NOZEROS_nparticles(42, df_rec_noise, df_rec_signal, df_sim_signal)
#    #ALCOR_gridmatrix_total, nparticles_array = grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware_nparticles(42, df_rec_noise, df_rec_signal, df_sim_signal, binwidth,)
#    #ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS_acceptancefiltered(42, df_rec_noise, df_rec_signal, df_sim_signal)
#    #ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware_acceptancefiltered(42, df_rec_noise, df_rec_signal, df_sim_signal, binwidth,)
#    #ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS_timingaware(42, df_rec_noise, df_rec_signal,binwidth)
#    ALCOR_gridmatrix_total = grid_ALCOR_optimized_signalplusnoise_NOZEROS(42, df_rec_noise, df_rec_signal)
#    print(ALCOR_gridmatrix_total.shape)
#    #print(nparticles_array.shape)
#    #ALCOR_gridmatrix_total = ALCOR_gridmatrix_total.reshape(ALCOR_gridmatrix_total.shape[0],6,5,42*int(10.16/binwidth))
#    ALCOR_gridmatrix_total = ALCOR_gridmatrix_total.reshape(ALCOR_gridmatrix_total.shape[0],6,5,42)
#
#    input_signal_matrices.append(ALCOR_gridmatrix_total)
#
#    #input_nparticles_array.append(nparticles_array)
#
#
#    end = time.perf_counter()
#    print(f"Tempo totale: {end - start:.4f}s")

In [17]:
input_noise_matrices = []
y_noise_training = []
for f in range(0,len(df_rec_signal_list)):
    print("NOISE FILE n.%d" % (f))
    start = time.perf_counter()
    df_rec_noise     = pd.read_csv(f"/content/drive/MyDrive/project_files/Noise_files/with_time/csv_reco/only_noise_reco_{f}_{FREQ}kHz_10ns.csv")
    #df_rec_noise     = pd.read_csv("../csv/2501_noise_events_300kHz_10ns/rec_hits.csv_%01d.csv" % (f+1), sep = ',')
    ALCOR_gridmatrix = grid_ALCOR_optimized_timingaware(42, df_rec_noise,binwidth)
    #ALCOR_gridmatrix = grid_ALCOR_optimized(42, df_rec_noise)
    #ALCOR_gridmatrix = ALCOR_gridmatrix.reshape(1000,6,5,42)
    ALCOR_gridmatrix = ALCOR_gridmatrix.reshape(1000,6,5,42*int(ten/binwidth)) #badile: era *20 invece che *4

    input_noise_matrices.append(ALCOR_gridmatrix)
    end = time.perf_counter()
    print(f"Tempo totale: {end - start:.4f}s")

NOISE FILE n.0


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/project_files/Noise_files/with_time/csv_reco/only_noise_reco_0_150kHz_10ns.csv'

In [ ]:
cosa_inutile=False
if cosa_inutile==True:
 for f in range(0,num_signal_ROOT_files-1):
    expected = set(range(1000))
    present = set(df_rec_signal_list[f]["event"])

    missing = expected - present

    if len(missing) == 0:
        print("Tutti gli eventi da 0 a 999 sono presenti")
    else:
        print(f"Mancano {len(missing)} eventi:", sorted(missing))

In [1]:
X_signal = np.array(input_signal_matrices)
print(X_signal.shape)
X_signal = X_signal.reshape(X_signal.shape[0]*X_signal.shape[1],30, 42*int(ten/binwidth))
#X_signal = X_signal.reshape(X_signal.shape[0]*X_signal.shape[1],30,42)
print(X_signal.shape)

NameError: name 'np' is not defined

In [ ]:
X_noise = np.array(input_noise_matrices)
print(X_noise.shape)
X_noise = X_noise.reshape(X_noise.shape[0]*X_noise.shape[1],30, 42*int(ten/binwidth))#badile: qui era signal
#X_noise = X_noise.reshape(X_noise.shape[0]*X_noise.shape[1],30,42)
print(X_noise.shape)

In [ ]:
# Supponiamo che il padding sia 0
mask_not_empty = np.any(X_signal != 0, axis=(1,2))  # True se l'evento ha almeno un valore non zero

# Filtriamo solo gli eventi non vuoti
X_signal_not_empty = X_signal[mask_not_empty]

print(X_signal_not_empty.shape)


In [ ]:
print(X_signal_not_empty.shape)

In [ ]:
import os

#directory di destinazione
target_dir = "/content/drive/MyDrive/project_files/input_time/"

# Verifichimo che la cartella esista, altrimenti la creiamo
os.makedirs(target_dir, exist_ok=True)

# 3. Salvataggio dei file
with open(f"{target_dir}signal_data_{FREQ}kHz.npy", "wb") as f:
    np.save(f, X_signal)

with open(f"{target_dir}noise_data_{FREQ}kHz.npy", "wb") as f:
    np.save(f, X_noise)

In [ ]:

with open(f'signal_data_{FREQ}kHz_timingaware.npy', 'wb') as f:
    np.save(f, X_signal)

with open(f'noise_data_{FREQ}kHz_timingaware.npy', 'wb') as f:
    #np.save(f, X_signal_not_empty)
    np.save(f, X_noise)

In [ ]:
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
